In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound
import os
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=7)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d')
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')
timestamp_run = peru_time.strftime('%d_%m_%Y_%H%M%S')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")


--- Fecha de inicio: 2026-06-05 ---
--- Fecha de Fin: 2026-06-12 ---
--- Fecha del proceso: 2026-06-12 00:00:00 ---


In [ ]:
## Variables de fecha como DataEntry
#var_fecha_ini = '2026-06-08' ## Desde cuando se debe traer la info - 7 dias atraz de un lunes.
#var_fecha_fin = '2026-06-15' ## Fecha del dia a ejecutar , deberia ser lunes -- fecha del nombre del archivo
#fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio 1: {var_fecha_ini} ---")
print(f"--- Fecha de Fin 2: {var_fecha_fin} ---")
print(f"--- Fecha del proceso 3: {fecha_fin_dt} ---")

--- Fecha de inicio 1: 2026-06-05 ---
--- Fecha de Fin 2: 2026-06-12 ---
--- Fecha del proceso 3: 2026-06-12 00:00:00 ---


In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')
var_mes = fecha_fin_dt.strftime('%m')
#var_fecha_file = fecha_fin_dt.strftime('%Y%m%d')
var_fecha_file = datetime.now().strftime('%Y%m%d')

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")

--- Año del proceso 2026 ---
--- Mes del proceso: 06 ---
--- Fecha del archivo: 20260612 ---


In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Operaciones/Desafiliaciones/{var_anho}/{var_mes}/"
nombre_final = f"Desafiliaciones_{var_fecha_file}.csv"  # <-- CSV
proyecto_operation = "prd-izipay-data-operation"
proyecto_storage = "prd-izipay-data-storage-pv"
proyecto_sensitivo = "prd-izipay-data-sensitive"
dataset_master_pii = "master_pii"


In [ ]:
# =============================================================================
# 4. Creación de Tabla Temporal en BigQuery
# =============================================================================

temp_table_id = f"{proyecto_operation}.master_stage_financial.temp_m_comercio_desafiliaciones_{var_fecha_file}"

query_temp = f"""
CREATE OR REPLACE TABLE `{temp_table_id}` as
with aux_iden_party_data_control as (
select
 party_id_izi,
 document_number
from {proyecto_sensitivo}.{dataset_master_pii}.iden_party_data_control
qualify row_number() over (partition by party_id_izi order by document_number desc ) = 1
)
SELECT
CAST(7963 AS INT64) AS acquirer_ica
,case
  when tipo_documento = 'RUC' then 'N'
  else 'Y'
end as sole_proprietorship
,null AS state_tax_id
,b.document_number AS national_tax_id
,trim(nom_comercio) AS merchant_name
,cod_comercio AS merchant_id
,case
    when cod_facilitador = '0' then null
    else trim(cast(cod_facilitador as string))
end as sub_merchant_id
,trim(nom_comercio) AS doing_business_as
,cod_giro_comercio AS merchant_category_code
,format_date('%Y-%m-%d',fecha_apertura_comercio) as date_opened
,format_date('%Y-%m-%d',fecha_bloqueo_comercio) as date_closed
--,substr(trim(AEAD.DECRYPT_STRING(c.key, a.direccion_comercio, c.constant)),1,101) AS business_address_1
,REGEXP_REPLACE(SUBSTR(TRIM(AEAD.DECRYPT_STRING(c.key, a.direccion_comercio, c.constant)), 1, 100),r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]','') AS business_address_1
,null AS business_address_2
,'Y' AS is_other_city
--,distrito_comercio AS city
,REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(
                TRIM(distrito_comercio),
                r'[ÁáÀàÄäÂâÃã]', 'A'),
              r'[ÉéÈèËëÊê]', 'E'),
            r'[ÍíÌìÏïÎî]', 'I'),
          r'[ÓóÒòÖöÔôÕõ]', 'O'),
        r'[ÚúÙùÜüÛû]', 'U'),
      r'[Ññ]', 'N'),
    r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]', '') AS city
,'PER' AS country
,substr(departamento_comercio,1,3) AS state_province
,cod_postal_comercio AS zip_postal_code
,trim(AEAD.DECRYPT_STRING(d.key, a.telefono_comercio, d.constant)) AS phone_number
,null AS alternate_phone_number
,null AS merchant_website
,'N' AS transaction_laundering
,'05' AS reason_code
,null AS adc_event_case_id
--,trim(REPLACE(AEAD.DECRYPT_STRING(e.key, a.nom_representante_legal, e.constant), '/', ' ')) AS principal1_first_name
,REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(TRIM(REPLACE(AEAD.DECRYPT_STRING(e.key, a.nom_representante_legal, e.constant), '/', ' ')),
              r'[ÁáÀàÄäÂâÃã]', 'A'),
            r'[ÉéÈèËëÊê]', 'E'),
          r'[ÍíÌìÏïÎî]', 'I'),
        r'[ÓóÒòÖöÔôÕõ]', 'O'),
      r'[ÚúÙùÜüÛû]', 'U'),
    r'[Ññ]', 'N'),
  r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]', '') AS principal1_first_name
,null AS principal1_middle_initial
,REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(TRIM(REPLACE(AEAD.DECRYPT_STRING(e.key, a.nom_representante_legal, e.constant), '/', ' ')),
              r'[ÁáÀàÄäÂâÃã]', 'A'),
            r'[ÉéÈèËëÊê]', 'E'),
          r'[ÍíÌìÏïÎî]', 'I'),
        r'[ÓóÒòÖöÔôÕõ]', 'O'),
      r'[ÚúÙùÜüÛû]', 'U'),
    r'[Ññ]', 'N'),
  r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]', '') AS principal1_last_name
,REGEXP_REPLACE(
  REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              SUBSTR(TRIM(AEAD.DECRYPT_STRING(c.key, a.direccion_comercio, c.constant)), 1, 100),
            r'[ÁáÀàÄäÂâÃã]', 'A'),   -- A con tilde/acento
          r'[ÉéÈèËëÊê]', 'E'),        -- E con tilde/acento
        r'[ÍíÌìÏïÎî]', 'I'),          -- I con tilde/acento
      r'[ÓóÒòÖöÔôÕõ]', 'O'),          -- O con tilde/acento
    r'[ÚúÙùÜüÛû]', 'U'),              -- U con tilde/acento
  r'[Ññ]', 'N'),                       -- Ñ
r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]', '')
AS principal1_address_1
,null AS principal1_address_2
,'Y' AS principal1_is_other_city
--,trim(a.distrito_comercio) AS principal1_city
,REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(TRIM(a.distrito_comercio),
              r'[ÁáÀàÄäÂâÃã]', 'A'),
            r'[ÉéÈèËëÊê]', 'E'),
          r'[ÍíÌìÏïÎî]', 'I'),
        r'[ÓóÒòÖöÔôÕõ]', 'O'),
      r'[ÚúÙùÜüÛû]', 'U'),
    r'[Ññ]', 'N'),
  r'[^A-Za-z0-9 !@#$^*()|:{{}}\\[\\]\\-=,./~`]', '') AS principal1_city
,'PER' AS principal1_country
--,TRIM(departamento_comercio) AS principal1_state_province
--,LEFT(TRIM(departamento_comercio), 3) AS principal1_state_province
,'PER' AS principal1_state_province
,a.cod_postal_comercio AS principal1_zip_postal_code
,trim(AEAD.DECRYPT_STRING(d.key, a.telefono_comercio, d.constant))  AS principal1_phone_number
,null AS principal1_alternate_phone_number
--,COALESCE(NULLIF(TRIM(REPLACE(AEAD.DECRYPT_STRING(f.key, a.correo_representante_legal, f.constant), '_', '')), ''), 'SINEMAIL@NA.COM') AS principal1_email_address
,CASE
    WHEN REGEXP_CONTAINS(TRIM(REPLACE(AEAD.DECRYPT_STRING(f.key, a.correo_representante_legal, f.constant), '_', '')),r'@') THEN TRIM(REPLACE(AEAD.DECRYPT_STRING(f.key, a.correo_representante_legal, f.constant), '_', ''))
ELSE 'SINEMAIL@NA.COM' END AS principal1_email_address
,null AS principal1_driver_s_license_number
,null AS principal1_driver_s_license_country
,null AS principal1_driver_s_license_state
,'1900-01-01' AS principal1_date_of_birth
,null AS principal1_national_id_ssn
,null AS principal2_first_name
,null AS principal2_middle_initial
,null AS principal2_last_name
,null AS principal2_address_1
,null AS principal2_address_2
,null AS principal2_is_other_city
,null AS principal2_city
,null AS principal2_country
,null AS principal2_state_province
,null AS principal2_zip_postal_code
,null AS principal2_phone_number
,null AS principal2_alternate_phone_number
,null AS principal2_email_address
,null AS principal2_driver_s_license_number
,null AS principal2_driver_s_license_country
,null AS principal2_driver_s_license_state
,null AS principal2_date_of_birth
,null AS principal2_national_id_ssn
,null AS principal3_first_name
,null AS principal3_middle_initial
,null AS principal3_last_name
,null AS principal3_address_1
,null AS principal3_address_2
,null AS principal3_is_other_city
,null AS principal3_city
,null AS principal3_country
,null AS principal3_state_province
,null AS principal3_zip_postal_code
,null AS principal3_phone_number
,null AS principal3_alternate_phone_number
,null AS principal3_email_address
,null AS principal3_driver_s_license_number
,null AS principal3_driver_s_license_country
,null AS principal3_driver_s_license_state
,null AS principal3_date_of_birth
,null AS principal3_national_id_ssn
,null AS principal4_first_name
,null AS principal4_middle_initial
,null AS principal4_last_name
,null AS principal4_address_1
,null AS principal4_address_2
,null AS principal4_is_other_city
,null AS principal4_city
,null AS principal4_country
,null AS principal4_state_province
,null AS principal4_zip_postal_code
,null AS principal4_phone_number
,null AS principal4_alternate_phone_number
,null AS principal4_email_address
,null AS principal4_driver_s_license_number
,null AS principal4_driver_s_license_country
,null AS principal4_driver_s_license_state
,null AS principal4_date_of_birth
,null AS principal4_national_id_ssn
,null AS principal5_first_name
,null AS principal5_middle_initial
,null AS principal5_last_name
,null AS principal5_address_1
,null AS principal5_address_2
,null AS principal5_is_other_city
,null AS principal5_city
,null AS principal5_country
,null AS principal5_state_province
,null AS principal5_zip_postal_code
,null AS principal5_phone_number
,null AS principal5_alternate_phone_number
,null AS principal5_email_address
,null AS principal5_driver_s_license_number
,null AS principal5_driver_s_license_country
,null AS principal5_driver_s_license_state
,null AS principal5_date_of_birth
,null AS principal5_national_id_ssn
from {proyecto_storage}.master_party.m_comercio a
left join aux_iden_party_data_control b on (b.party_id_izi = a.party_id_izi)
left join {proyecto_sensitivo}.secure_secrets.config_protected_data c on ( 1=1 and c.code = 'C_ADDRESS' )
left join {proyecto_sensitivo}.secure_secrets.config_protected_data d on ( 1=1 and d.code = 'C_TELEPHONE' )
left join {proyecto_sensitivo}.secure_secrets.config_protected_data e on ( 1=1 and e.code = 'C_FULL_NAME' )
left join {proyecto_sensitivo}.secure_secrets.config_protected_data f on ( 1=1 and f.code = 'C_EMAIL' )
where flag_bloqueo_fraude = TRUE
 and cod_comercio is not null
 and a.party_id_izi is not null
 and cod_giro_comercio is not null
 and fecha_apertura_comercio is not null
 and fecha_bloqueo_comercio >= DATE '{var_fecha_ini}';
"""

print(f"Creando tabla temporal: {temp_table_id} ...")
clientBQ.query(query_temp).result()
print("✅ Tabla temporal creada correctamente.")
print(f"Query Ejecutado:  ...")
print(f"{query_temp} ")

Creando tabla temporal: prd-izipay-data-operation.master_stage_financial.temp_m_comercio_desafiliaciones_20260612 ...
✅ Tabla temporal creada correctamente.
Query Ejecutado:  ...

CREATE OR REPLACE TABLE `prd-izipay-data-operation.master_stage_financial.temp_m_comercio_desafiliaciones_20260612` as
with aux_iden_party_data_control as (
select
 party_id_izi,
 document_number
from prd-izipay-data-storage-pv.mc2253.iden_party_data_control
qualify row_number() over (partition by party_id_izi order by document_number desc ) = 1
)
SELECT
CAST(7963 AS INT64) AS acquirer_ica
,case
  when tipo_documento = 'RUC' then 'N'
  else 'Y'
end as sole_proprietorship
,null AS state_tax_id
,b.document_number AS national_tax_id
,trim(nom_comercio) AS merchant_name
,cod_comercio AS merchant_id
,case
    when cod_facilitador = '0' then null
    else trim(cast(cod_facilitador as string))
end as sub_merchant_id
,trim(nom_comercio) AS doing_business_as
,cod_giro_comercio AS merchant_category_code
,format_date('%

In [ ]:
# =============================================================================
# 5. Exportación desde Tabla Temporal a GCS (CSV)
# =============================================================================
import time

uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{timestamp_run}_*.csv"
print(f"Exportando desde tabla temporal a: {uri_temporal} ...")

query_export = f"""
EXPORT DATA OPTIONS (
  uri      = '{uri_temporal}',
  format   = 'CSV',
  overwrite = true,
  header   = true
) AS
SELECT * FROM `{temp_table_id}`;
"""

# Polling con feedback cada 15s
job = clientBQ.query(query_export)
print(f"📋 Job iniciado: {job.job_id}")
t0 = time.time()

while True:
    job.reload()
    elapsed = int(time.time() - t0)
    mins, secs = divmod(elapsed, 60)
    if job.state == 'DONE':
        break
    print(f"   ⏳ {job.state} ... {mins}m {secs:02d}s transcurridos")
    time.sleep(15)

if job.errors:
    raise RuntimeError(f"❌ Error en EXPORT DATA: {job.errors}")

elapsed = int(time.time() - t0)
mins, secs = divmod(elapsed, 60)
print(f"✅ Exportación a GCS completada en {mins}m {secs:02d}s")

# Limpiar tabla temporal
clientBQ.delete_table(temp_table_id)
print(f"🧹 Tabla temporal eliminada: {temp_table_id}")


Exportando desde tabla temporal a: gs://adls-reportes/Operaciones/Desafiliaciones/2026/06/temp_12_06_2026_142130_*.csv ...
📋 Job iniciado: 21c42265-e495-4800-a3e7-ec156cf57833
   ⏳ RUNNING ... 0m 00s transcurridos
✅ Exportación a GCS completada en 0m 15s
🧹 Tabla temporal eliminada: prd-izipay-data-operation.master_stage_financial.temp_m_comercio_desafiliaciones_20260612


In [ ]:
# =============================================================================
# 6. Consolidación incremental a CSV comprimido (sin cargar todo en memoria)
# =============================================================================
# Escribe part a part en un archivo local temporal y luego sube a GCS.
# Así evitamos acumular todo el dataset en RAM.

print("Consolidando archivos CSV en uno solo (modo incremental)...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{timestamp_run}_"

blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar.")
else:
    local_tmp = f"/tmp/{nombre_final}"
    primera_parte = True

    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        #df_part = pd.read_csv(uri_parte)
        df_part = pd.read_csv(uri_parte, dtype=str, keep_default_na=False)
        df_part.rename(columns={'acquirer_ica': 'Acquirer ICA', 'sole_proprietorship': 'Sole Proprietorship', 'state_tax_id': 'State Tax ID', 'national_tax_id': 'National Tax ID', 'merchant_name': 'Merchant Name', 'merchant_id': 'Merchant ID', 'sub_merchant_id': 'Sub Merchant ID', 'doing_business_as': 'Doing Business As', 'merchant_category_code': 'Merchant Category Code', 'date_opened': 'Date Opened', 'date_closed': 'Date Closed', 'business_address_1': 'Business Address 1', 'business_address_2': 'Business Address 2', 'is_other_city': 'Is Other City', 'city': 'City', 'country': 'Country', 'state_province': 'State/Province', 'zip_postal_code': 'Zip/Postal Code', 'phone_number': 'Phone Number', 'alternate_phone_number': 'Alternate Phone Number', 'merchant_website': 'Merchant Website', 'transaction_laundering': 'Transaction Laundering', 'reason_code': 'Reason code', 'adc_event_case_id': 'ADC Event Case ID', 'principal1_first_name': 'Principal(1) First Name', 'principal1_middle_initial': 'Principal(1) Middle Initial', 'principal1_last_name': 'Principal(1) Last Name', 'principal1_address_1': 'Principal(1) Address 1', 'principal1_address_2': 'Principal(1) Address 2', 'principal1_is_other_city': 'Principal(1) Is Other City', 'principal1_city': 'Principal(1) City', 'principal1_country': 'Principal(1) Country', 'principal1_state_province': 'Principal(1) State/Province', 'principal1_zip_postal_code': 'Principal(1) Zip/Postal Code', 'principal1_phone_number': 'Principal(1) Phone Number', 'principal1_alternate_phone_number': 'Principal(1) Alternate Phone Number', 'principal1_email_address': 'Principal(1) Email Address', 'principal1_driver_s_license_number': "Principal(1) Driver's License Number", 'principal1_driver_s_license_country': "Principal(1) Driver's License Country", 'principal1_driver_s_license_state': "Principal(1) Driver's License State", 'principal1_date_of_birth': 'Principal(1) Date of Birth', 'principal1_national_id_ssn': 'Principal(1) National ID/SSN', 'principal2_first_name': 'Principal(2) First Name', 'principal2_middle_initial': 'Principal(2) Middle Initial', 'principal2_last_name': 'Principal(2) Last Name', 'principal2_address_1': 'Principal(2) Address 1', 'principal2_address_2': 'Principal(2) Address 2', 'principal2_is_other_city': 'Principal(2) Is Other City', 'principal2_city': 'Principal(2) City', 'principal2_country': 'Principal(2) Country', 'principal2_state_province': 'Principal(2) State/Province', 'principal2_zip_postal_code': 'Principal(2) Zip/Postal Code', 'principal2_phone_number': 'Principal(2) Phone Number', 'principal2_alternate_phone_number': 'Principal(2) Alternate Phone Number', 'principal2_email_address': 'Principal(2) Email Address', 'principal2_driver_s_license_number': "Principal(2) Driver's License Number", 'principal2_driver_s_license_country': "Principal(2) Driver's License Country", 'principal2_driver_s_license_state': "Principal(2) Driver's License State", 'principal2_date_of_birth': 'Principal(2) Date of Birth', 'principal2_national_id_ssn': 'Principal(2) National ID/SSN', 'principal3_first_name': 'Principal(3) First Name', 'principal3_middle_initial': 'Principal(3) Middle Initial', 'principal3_last_name': 'Principal(3) Last Name', 'principal3_address_1': 'Principal(3) Address 1', 'principal3_address_2': 'Principal(3) Address 2', 'principal3_is_other_city': 'Principal(3) Is Other City', 'principal3_city': 'Principal(3) City', 'principal3_country': 'Principal(3) Country', 'principal3_state_province': 'Principal(3) State/Province', 'principal3_zip_postal_code': 'Principal(3) Zip/Postal Code', 'principal3_phone_number': 'Principal(3) Phone Number', 'principal3_alternate_phone_number': 'Principal(3) Alternate Phone Number', 'principal3_email_address': 'Principal(3) Email Address', 'principal3_driver_s_license_number': "Principal(3) Driver's License Number", 'principal3_driver_s_license_country': "Principal(3) Driver's License Country", 'principal3_driver_s_license_state': "Principal(3) Driver's License State", 'principal3_date_of_birth': 'Principal(3) Date of Birth', 'principal3_national_id_ssn': 'Principal(3) National ID/SSN', 'principal4_first_name': 'Principal(4) First Name', 'principal4_middle_initial': 'Principal(4) Middle Initial', 'principal4_last_name': 'Principal(4) Last Name', 'principal4_address_1': 'Principal(4) Address 1', 'principal4_address_2': 'Principal(4) Address 2', 'principal4_is_other_city': 'Principal(4) Is Other City', 'principal4_city': 'Principal(4) City', 'principal4_country': 'Principal(4) Country', 'principal4_state_province': 'Principal(4) State/Province', 'principal4_zip_postal_code': 'Principal(4) Zip/Postal Code', 'principal4_phone_number': 'Principal(4) Phone Number', 'principal4_alternate_phone_number': 'Principal(4) Alternate Phone Number', 'principal4_email_address': 'Principal(4) Email Address', 'principal4_driver_s_license_number': "Principal(4) Driver's License Number", 'principal4_driver_s_license_country': "Principal(4) Driver's License Country", 'principal4_driver_s_license_state': "Principal(4) Driver's License State", 'principal4_date_of_birth': 'Principal(4) Date of Birth', 'principal4_national_id_ssn': 'Principal(4) National ID/SSN', 'principal5_first_name': 'Principal(5) First Name', 'principal5_middle_initial': 'Principal(5) Middle Initial', 'principal5_last_name': 'Principal(5) Last Name', 'principal5_address_1': 'Principal(5) Address 1', 'principal5_address_2': 'Principal(5) Address 2', 'principal5_is_other_city': 'Principal(5) Is Other City', 'principal5_city': 'Principal(5) City', 'principal5_country': 'Principal(5) Country', 'principal5_state_province': 'Principal(5) State/Province', 'principal5_zip_postal_code': 'Principal(5) Zip/Postal Code', 'principal5_phone_number': 'Principal(5) Phone Number', 'principal5_alternate_phone_number': 'Principal(5) Alternate Phone Number', 'principal5_email_address': 'Principal(5) Email Address', 'principal5_driver_s_license_number': "Principal(5) Driver's License Number", 'principal5_driver_s_license_country': "Principal(5) Driver's License Country", 'principal5_driver_s_license_state': "Principal(5) Driver's License State", 'principal5_date_of_birth': 'Principal(5) Date of Birth', 'principal5_national_id_ssn': 'Principal(5) National ID/SSN'}, inplace=True)
        df_part.to_csv(
            local_tmp,
            mode='a',
            index=False,
            header=primera_parte,
            sep=','
        )
        primera_parte = False
        del df_part

    # Subir archivo consolidado a GCS
    ruta_final_full = f"{ruta_base}{nombre_final}"
    blob_final = bucket.blob(ruta_final_full)
    blob_final.upload_from_filename(local_tmp)
    os.remove(local_tmp)

    # Borrar parts temporales de GCS
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: gs://{bucket_name}/{ruta_final_full}")
    print(f"🧹 {len(blobs)} archivos temporales eliminados")


Consolidando archivos CSV en uno solo (modo incremental)...
✅ ÉXITO: Archivo único creado en: gs://adls-reportes/Operaciones/Desafiliaciones/2026/06/Desafiliaciones_20260612.csv
🧹 1 archivos temporales eliminados
